<!-- # High-Resolution Image Synthesis with Latent Diffusion Models -->
# 基于潜在扩散模型的超分辨率图像合成

## 摘要 Abstract

## 引言 Introduction

__贡献 contributions__:
1. **高维数据处理优势**：相较于纯Transformer方法，模型更易适配高维数据，能在更高保真度、更细节化的压缩水平下工作，且可高效应用于百万像素级图像的高分辨率合成。  
2. **性能与成本平衡**：在多任务（无条件图像合成、修复、随机超分辨率）及多数据集上表现优异，同时大幅降低计算成本，相较基于像素的扩散方法，推理成本也显著减少。  
3. **架构设计简化**：不同于需同时学习编解码器与基于分数先验的方法，无需精细权衡重建与生成能力，确保极高保真度的重建效果，且对 latent 空间正则化需求极低。  
4. **大尺寸图像生成能力**：针对超分辨率、修复等密集条件任务，可通过卷积方式生成约$1024^2$像素的大型连贯图像。  
5. **通用条件机制**：基于交叉注意力设计通用条件机制，支持多模态训练，已用于训练类别条件、文本到图像及布局到图像模型。  
6. **开源预训练模型**：发布 latent 扩散与自编码预训练模型，除扩散模型（DMs）训练外，还可复用至多种任务。

## 相关工作 Related Work


### 图像合成生成模型研究现状
1. **主流模型局限**：  
   - GAN：可高效生成高分辨率、高感知质量图像，但优化困难且难以捕捉完整数据分布。  
   - 似然基方法（VAE、流模型）：优化更稳定、支持高效合成，但样本质量不及GAN；自回归模型（ARM）密度估计性能强，但架构计算量大、采样过程串行，仅适用于低分辨率图像。  
   - 像素级表征问题：需花费大量模型容量建模细微高频细节，导致训练耗时，因此部分研究采用两阶段方法，用ARM建模压缩后的 latent 图像空间而非原始像素。  

2. **扩散概率模型（DM）进展与不足**：  
   - 优势：结合UNet骨干网络时适配图像数据归纳偏置，在密度估计和样本质量上达SOTA，通过重加权目标可权衡图像质量与压缩能力。  
   - 缺陷：像素空间训练推理速度慢、成本高，虽可通过采样策略和分层方法缓解推理问题，但高分辨率数据训练仍需高昂梯度计算成本。  

3. **本文方案**：提出潜扩散模型（LDMs），在低维压缩 latent 空间工作，大幅降低训练计算成本并加速推理，且几乎不损失合成质量。


### 两阶段图像合成研究现状
1. **核心思路**：融合不同生成方法的优势，通过两阶段架构提升效率与性能，主流方向为用特定模型学习压缩 latent 空间的先验。  
   - VQ-VAEs：基于离散 latent 空间，用ARM学习表达性先验，部分扩展至文本-图像生成任务；另有研究用条件可逆网络实现跨域 latent 空间转换。  
   - VQGANs：第一阶段采用对抗性和感知目标，助力自回归Transformer适配更大图像。  

2. **现有缺陷**：为使ARM训练可行需高压缩率，导致模型性能受限；而降低压缩率则会因ARM参数规模（数十亿级）带来极高计算成本。  

3. **本文优势**：LDMs凭借卷积骨干网络，可更平缓地适配高维 latent 空间，无需在压缩率上妥协，能在优化第一阶段性能的同时，保证高保真重建效果，避免上述权衡问题。  
   - 对比其他方法：部分研究联合/单独学习编解码模型与基于分数的先验，但前者需平衡重建与生成能力且性能不及LDMs，后者仅适用于人脸等结构规整的图像。

## 方法 Method

本文提出**潜扩散模型（LDMs）**，核心思路是将**感知压缩**与**生成建模**两个阶段解耦，在低维潜空间中训练扩散模型，以降低高分辨率图像合成的计算成本。整体方法分为三个关键部分：

1.  **感知图像压缩 Perceptual Image Compression** 
    构建基于自动编码器的感知压缩模型，训练目标结合**感知损失**与**基于块的对抗损失**，避免仅用像素损失（如$L_1/L_2$）导致的重建模糊问题，确保重建图像贴合真实图像流形。
    - 编码过程：编码器$\mathcal{E}$将 RGB 图像$x\in \mathbb{R}^{H\times W \times 3}$下采样为低维潜表示$z=\mathcal{E}(x)$，下采样因子$f=2^m$（$m$为自然数）；解码器$\mathcal{D}$再将$z$重建为图像$\tilde{x}=\mathcal{D}(z)=\mathcal{D}(\mathcal{E}(x))$。
    - 潜空间正则化：采用两种方式避免潜空间方差过大——一是KL正则化（向标准正态分布施加KL惩罚，类似VAE）；二是向量量化（VQ）正则化（在解码器中嵌入向量量化层，等效于VQGAN）。
    - 优势：凭借扩散模型对空间结构的适配性，仅需**温和压缩率**即可实现高保真重建，避免了以往方法为适配自回归模型而采用的有损高压缩策略。

2.  **潜扩散模型构建 Latent Diffusion Models**
    扩散模型的核心是学习数据分布的逆向去噪过程，标准训练目标为预测噪声的$L_2$损失：
    $$L_{DM} = \mathbb{E}_{x, \epsilon \sim \mathcal{N}(0, 1),  t }\Big[ \Vert \epsilon - \epsilon_\theta(x_{t},t) \Vert_{2}^{2}\Big]$$
    本文将扩散模型的训练空间从高维像素空间迁移至**低维潜空间**，具体改进如下：
    - 训练目标调整为对潜表示$z=\mathcal{E}(x)$的噪声预测损失：
      $$L_{LDM} := \mathbb{E}_{\mathcal{E}(x), \epsilon \sim \mathcal{N}(0, 1),  t}\Big[ \Vert \epsilon - \epsilon_\theta(z_{t},t) \Vert_{2}^{2}\Big]$$
    - 模型骨干采用**时间条件UNet**，以2D卷积层为主，充分利用图像的空间归纳偏置；
    - 优势：模型无需建模像素空间的高频无关细节，聚焦语义信息，大幅降低训练与推理的计算成本；生成时只需将潜空间采样结果输入解码器，即可得到图像。

3.  **通用条件机制 Conditioning Mechanisms**
    为实现可控图像生成，通过**交叉注意力机制**增强UNet骨干的条件建模能力，支持多模态条件输入。
    - 条件编码：设计域专用编码器$\tau_\theta$，将文本、语义图等不同模态的条件输入$y$映射为中间表示$\tau_\theta(y)\in \mathbb{R}^{M \times d}$；
    - 交叉注意力融合：将UNet中间层特征作为查询（$Q$），将$\tau_\theta(y)$作为键（$K$）和值（$V$），通过注意力机制实现条件信息与潜特征的融合；
    $$\text{Attention}(Q, K, V) = \text{softmax}\Big(\frac{QK^T}{\sqrt{d}}\Big)V$$
    $$Q = W_Q^{(i)}\cdot \varphi_i(z_t), \quad K = W_K^{(i)}\cdot \tau_\theta(y), \quad V = W_V^{(i)}\cdot \tau_\theta(y)$$
    - 其中：
        - $\varphi_i(z_t)\in \mathbb{R}^{N\times d_\epsilon^i}$为UNet $\epsilon_\theta$ 第$i$层的中间特征（flattened）；
        - $W_V^{(i)} \in \mathbb{R}^{d\times d_\epsilon^i}, W_Q^{(i)} \in \mathbb{R}^{d\times d_\tau}, W_K^{(i)} \in \mathbb{R}^{d \times d_\tau}$为线性映射矩阵。
    - 条件训练目标：联合优化条件编码器与扩散模型，损失函数为：
      $$L_{LDM} := \mathbb{E}_{\mathcal{E}(x), y, \epsilon \sim \mathcal{N}(0, 1), t }\Big[ \Vert \epsilon - \epsilon_\theta(z_{t},t, \tau_\theta(y)) \Vert_{2}^{2}\Big]$$
    - 灵活性：可通过替换$\tau_\theta$适配不同任务，如类别条件生成、文本到图像生成、布局到图像生成等。
整体而言，该方法通过**阶段解耦+潜空间建模+交叉注意力条件机制**，在保证合成质量的前提下，大幅提升了扩散模型在高分辨率图像生成任务中的计算效率与灵活性。


<div style="background-color:white;margin: auto; border-radius:5px; width: 80%">
    <image src="./assets/final_figure.png" alt="Method Overview" >
    <p style="text-align:center; font-size:12px; color:#555;">图3. 我们通过连接或更一般的交叉注意力机制来对 LDMs 进行条件化。</p>
</div>

- 数据流:
    - 输入的初始形态
        - 文本 --(CLIP) --> (B, L, 768) -- B: bathch size, L: token length,
        - 噪声(潜向量) --> (B, 16, 64, 64) -- 16: channels, 64x64: spatial size
    - 第一步: 将(16, 64, 64)转化为序列特征
        - SpatialTransformer:
            - 输入: (B, 16, 64, 64)
            - Unet input_blocks升维: (B, 448, 64, 64) 
            - flatten输出: (B, 4096, 448)
    - 第二步: 融合文本特征与图像特征（交叉注意力，维度对齐）
        - 注意力机制：
            - Q(噪声): 图像特征(B, 4096, 448) --线性映射-> Q(B, 4096, 448) -8头- > (B, 8, 4096, 56)
            - K(文本): 文本特征(B, L, 768) --线性映射--> K(B, L, 448) -8头- > (B, 8, L, 56)
            - V(文本): 文本特征(B, L, 768) --线性映射--> V(B, L, 448) -8头- > (B, 8, L, 56)
            - $Q^i\times K^{iT}$ : (B, 1|8, 4096, L) * 8个头
            - $\text{softmax}{\frac{Q^i\times K^{iT}}{\sqrt{d}}} V$ : (B, 1|8, 4096, 56) * 8个头
            - 拼接8头输出: (B, 4096, 448)
    - 第三步: 输出还原为2D特征图
        - (B, 4096, 448) --线性映射--> (B, 448, 64, 64)
        - 后续：通过SpatialTransformer的proj_out层(Conv2d(448 --> 448))，与UNet的CNN分支特征融合，继续去噪过程。
    - 第四步: U-Net在不同下采样/上采样阶段，噪声的通道数会变化(448 -> 896 -> 1344 -> 1792)，但交叉注意力的逻辑完全一致: 
        - 阶段1: to_k/to_v: Linear(768 --> 448)
        - 阶段2: to_k/to_v: Linear(768 --> 896)
        - 阶段3: to_k/to_v: Linear(768 --> 1344)
        - 阶段4: to_k/to_v: Linear(768 --> 1792)